# 02 -- Validating Graph Data

The core use case of orthograph: validate that a concrete set of nodes and
relationships conforms to a declared graph data model.

`GraphValidator` checks:
- **Node / relationship labels** -- only labels declared in the model are accepted.
- **Property schemas** -- required fields, types, and no extra properties.
- **Referential integrity** -- every relationship endpoint must point to an existing node.
- **Cardinality** -- each node has the right number of relationships.
- **Entity presence** -- required (non-optional) types must have at least one instance.

This notebook walks through each of these checks using the filmography domain.

In [ ]:
from typing import Optional

from orthograph.diagnostics.result import GraphValidationError
from orthograph.graph_definition.graph_definition import GraphDefinition
from orthograph.graph_definition.models import Cardinality, NodeModel, RelationshipModel
from orthograph.graph_definition.validation import GraphValidator


# -- Node types --


class Person(NodeModel):
    __label__ = "Person"
    __uid_field__ = "name"
    name: str
    age: int
    email: Optional[str] = None


class Movie(NodeModel):
    __label__ = "Movie"
    __uid_field__ = "title"
    title: str
    year: int
    rating: Optional[float] = None


class City(NodeModel):
    __label__ = "City"
    __uid_field__ = "name"
    name: str
    country: str


# -- Relationship types --


class ActedIn(RelationshipModel):
    __label__ = "ACTED_IN"
    __source_label__ = "Person"
    __target_label__ = "Movie"
    role: str


class Directed(RelationshipModel):
    __label__ = "DIRECTED"
    __source_label__ = "Person"
    __target_label__ = "Movie"


class LivesIn(RelationshipModel):
    __label__ = "LIVES_IN"
    __source_label__ = "Person"
    __target_label__ = "City"
    __source_cardinality__ = Cardinality.ONE
    __target_cardinality__ = Cardinality.ZERO_OR_MORE


# -- Model --

graph_definition = GraphDefinition(
    name="Filmography",
    node_types=[Person, Movie, City],
    relationship_types=[ActedIn, Directed, LivesIn],
)

validator = GraphValidator(graph_definition)
print(
    f"Model created. \n Node labels: {graph_definition.node_labels} \n Relationship labels: {graph_definition.relationship_labels}"
)

## Validating Individual Nodes

Node data is passed as a list of dicts. Each dict must include a `__label__` key
that identifies the node type, plus property fields matching the model definition.

`validate_nodes()` checks labels, required properties, types, and extra properties
without performing referential integrity or cardinality checks.

In [ ]:
valid_nodes = [
    {"__label__": "Person", "name": "Alice", "age": 32, "email": "alice@example.com"},
    {"__label__": "Person", "name": "Bob", "age": 28},
    {"__label__": "Movie", "title": "Inception", "year": 2010, "rating": 8.8},
    {"__label__": "City", "name": "London", "country": "UK"},
]

result = validator.validate_nodes(valid_nodes)
print("is_valid:", result.is_valid)
print("errors:  ", len(result.errors))

## Catching Validation Errors

The validator catches several categories of node-level problems:

| Code | Meaning |
|---|---|
| `MISSING_LABEL` | The dict has no `__label__` key. |
| `UNKNOWN_NODE_LABEL` | The label is not registered in the model. |
| `EXTRA_PROPERTIES` | The dict contains property keys not declared on the model. |
| `PROPERTY_VALIDATION_ERROR` | A required property is missing, or a value has the wrong type. |

In [ ]:
bad_nodes = [
    # 1. Unknown label
    {"__label__": "Animal", "species": "Dog"},
    # 2. Missing __label__ entirely
    {"name": "Alice", "age": 30},
    # 3. Missing required property (age)
    {"__label__": "Person", "name": "Charlie"},
    # 4. Wrong type (age should be int)
    {"__label__": "Person", "name": "Diana", "age": "twenty-five"},
    # 5. Extra property not in the model
    {"__label__": "Movie", "title": "Tenet", "year": 2020, "budget": 200_000_000},
]

result = validator.validate_nodes(nodes=bad_nodes)
print(f"Found {len(result.errors)} errors:\n")
for err in result.errors:
    print(f"  [{err.code}] {err.message}")

## Validating Relationships

Relationship data follows the same dict convention, with three reserved keys:

- `__label__` -- the relationship type name.
- `__source_uid__` -- the UID of the source node.
- `__target_uid__` -- the UID of the target node.

Any additional keys are treated as relationship properties.

`validate_relationships()` checks labels, required properties, and structural
completeness (both endpoints must be specified). It does **not** check whether
the referenced nodes actually exist -- that is referential integrity, covered later.

In [ ]:
# Valid relationships (structurally)
valid_rels = [
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Alice",
        "__target_uid__": "Inception",
        "role": "Cobb",
    },
    {"__label__": "DIRECTED", "__source_uid__": "Bob", "__target_uid__": "Inception"},
]

result = validator.validate_relationships(relationships=valid_rels)
print("Valid relationships -- is_valid:", result.is_valid)
print()

In [ ]:
# Invalid relationships
bad_rels = [
    # Unknown label
    {"__label__": "PRODUCED", "__source_uid__": "Alice", "__target_uid__": "Inception"},
    # Missing endpoint
    {"__label__": "DIRECTED", "__source_uid__": "Bob"},
    # Missing required property 'role'
    {"__label__": "ACTED_IN", "__source_uid__": "Alice", "__target_uid__": "Inception"},
]

result = validator.validate_relationships(relationships=bad_rels)
print(f"Bad relationships -- {len(result.errors)} errors:")
for err in result.errors:
    print(f"  [{err.code}] {err.message}")

## Full Graph Validation

The main entry point is `validator.validate(nodes, relationships)`. It runs
all checks in sequence: node validation, relationship validation, referential
integrity, cardinality, and entity presence.

In [ ]:
nodes = [
    {"__label__": "Person", "name": "Alice", "age": 32},
    {"__label__": "Person", "name": "Bob", "age": 28},
    {"__label__": "Movie", "title": "Inception", "year": 2010},
    {"__label__": "Movie", "title": "Tenet", "year": 2020},
    {"__label__": "City", "name": "London", "country": "UK"},
    {"__label__": "City", "name": "Los Angeles", "country": "US"},
]

relationships = [
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Alice",
        "__target_uid__": "Inception",
        "role": "Cobb",
    },
    {"__label__": "DIRECTED", "__source_uid__": "Bob", "__target_uid__": "Tenet"},
    {"__label__": "LIVES_IN", "__source_uid__": "Alice", "__target_uid__": "London"},
    {"__label__": "LIVES_IN", "__source_uid__": "Bob", "__target_uid__": "Los Angeles"},
]

result = validator.validate(nodes=nodes, relationships=relationships)
print("Full graph validation")
print("  is_valid:", result.is_valid)
print("  errors:  ", len(result.errors))
print("  warnings:", len(result.warnings))

## Referential Integrity

When running full validation, the validator checks that every `__source_uid__` and
`__target_uid__` in a relationship points to an actual node in the data.

Two types of referential problems are detected:

- **Dangling reference** (`DANGLING_REFERENCE`) -- a relationship endpoint points to a
  UID that does not exist in the provided nodes.
- **Wrong endpoint type** (`WRONG_ENDPOINT_TYPE`) -- a relationship endpoint points to a
  node that exists but has the wrong label (e.g. ACTED_IN source is a Movie, not a Person).

In [ ]:
# Dangling reference: "Ghost" does not exist in nodes
nodes_partial = [
    {"__label__": "Movie", "title": "Inception", "year": 2010},
]
rels_dangling = [
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "Ghost",
        "__target_uid__": "Inception",
        "role": "X",
    },
]

result = validator.validate(nodes_partial, rels_dangling)
print("Dangling reference:")
for err in result.errors:
    print(f"  [{err.code}] {err.message}")

print()

# Wrong endpoint type: source is a Movie, but ACTED_IN expects Person
nodes_wrong_type = [
    {"__label__": "Movie", "title": "A", "year": 2020},
    {"__label__": "Movie", "title": "B", "year": 2021},
]
rels_wrong_type = [
    {
        "__label__": "ACTED_IN",
        "__source_uid__": "A",
        "__target_uid__": "B",
        "role": "Y",
    },
]

result = validator.validate(nodes_wrong_type, rels_wrong_type)
print("Wrong endpoint type:")
for err in result.errors:
    print(f"  [{err.code}] {err.message}")

## Working with Validation Results

`ValidationResult` provides structured access to all issues:

- `result.is_valid` -- `True` if there are zero errors.
- `result.errors` -- list of `ValidationIssue` objects with severity ERROR.
- `result.warnings` -- list of `ValidationIssue` objects with severity WARNING.
- `result.issues` -- all issues regardless of severity.
- `result.raise_on_errors()` -- raises `GraphValidationError` if any errors exist.

Each `ValidationIssue` has a `code`, `severity`, `entity_type`, `entity_id`,
`message`, and an optional `context` dict with structured details.

In [ ]:
# Build a result with both errors and valid data mixed in
mixed_nodes = [
    {"__label__": "Person", "name": "Alice", "age": 32},
    {"__label__": "Person", "name": "Bad"},  # missing 'age'
    {"__label__": "Unknown"},  # unknown label
]

result = validator.validate_nodes(mixed_nodes)

print("is_valid:", result.is_valid)
print()

# Iterate individual errors
print("Errors:")
for issue in result.errors:
    print(f"  code={issue.code}  entity={issue.entity_id}")
    print(f"    {issue.message}")

print()
print("Warnings:", result.warnings)

print()

# raise_on_errors() raises GraphValidationError
try:
    result.raise_on_errors()
except GraphValidationError as exc:
    print(f"raise_on_errors() raised with {len(exc.issues)} issue(s)")

## Undirected Relationship Validation

Undirected relationships (`__directed__ = False`) are symmetric -- the validator
accepts endpoint data stored in either direction. This is important when loading data
from a database where the internal storage direction may be arbitrary.

For undirected relationships, cardinality is checked against the **total** count of
relationships connected to a node (both outgoing and incoming combined), rather than
separately.

In [ ]:
# Define an undirected cross-type relationship
class Company(NodeModel):
    __label__ = "Company"
    __uid_field__ = "name"
    name: str


class FriendOf(RelationshipModel):
    __label__ = "FRIEND_OF"
    __source_label__ = "Person"
    __target_label__ = "Person"
    __directed__ = False  # symmetric


class Collaborates(RelationshipModel):
    __label__ = "COLLABORATES"
    __source_label__ = "Person"
    __target_label__ = "Company"
    __directed__ = False  # undirected, cross-type


social_model = GraphDefinition(
    name="Social",
    node_types=[Person, Company],
    relationship_types=[FriendOf, Collaborates],
)
social_validator = GraphValidator(social_model)
print("Social model created.")

In [ ]:
# Undirected same-type: both directions are valid
nodes = [
    {"__label__": "Person", "name": "Alice", "age": 30},
    {"__label__": "Person", "name": "Bob", "age": 25},
]

# Forward direction (as defined: Person -> Person)
rels_forward = [
    {"__label__": "FRIEND_OF", "__source_uid__": "Alice", "__target_uid__": "Bob"}
]
result = social_validator.validate(nodes=nodes, relationships=rels_forward)
print("FRIEND_OF Alice -> Bob:  is_valid =", result.is_valid)

# Reverse direction (Bob -> Alice, still valid because undirected)
rels_reverse = [
    {"__label__": "FRIEND_OF", "__source_uid__": "Bob", "__target_uid__": "Alice"}
]
result = social_validator.validate(nodes=nodes, relationships=rels_reverse)
print("FRIEND_OF Bob -> Alice:  is_valid =", result.is_valid)

In [ ]:
# Undirected cross-type: accepts either direction
nodes = [
    {"__label__": "Person", "name": "Alice", "age": 30},
    {"__label__": "Company", "name": "Acme"},
]

# Forward: Person -> Company (as defined)
rels_forward = [
    {"__label__": "COLLABORATES", "__source_uid__": "Alice", "__target_uid__": "Acme"}
]
result = social_validator.validate(nodes=nodes, relationships=rels_forward)
print("COLLABORATES Person -> Company:  is_valid =", result.is_valid)

# Reverse: Company -> Person (reversed from definition, still valid)
rels_reverse = [
    {"__label__": "COLLABORATES", "__source_uid__": "Acme", "__target_uid__": "Alice"}
]
result = social_validator.validate(nodes=nodes, relationships=rels_reverse)
print("COLLABORATES Company -> Person:  is_valid =", result.is_valid)

# But wrong types are still rejected
nodes_wrong = [
    {"__label__": "Person", "name": "Alice", "age": 30},
    {"__label__": "Person", "name": "Bob", "age": 25},
]
rels_wrong = [
    {"__label__": "COLLABORATES", "__source_uid__": "Alice", "__target_uid__": "Bob"}
]
result = social_validator.validate(nodes=nodes_wrong, relationships=rels_wrong)
print("COLLABORATES Person -> Person:   is_valid =", result.is_valid, "(wrong types)")